In [ ]:
! pip install yfinance requests pandas sqlalchemy anthropic -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 753.6/753.6 kB 16.2 MB/s eta 0:00:00


In [ ]:
from google.colab import userdata

NEWS_API_KEY = userdata.get('NEWS_API_KEY')w
print('NEWS API KEY LOADED' if NEWS_API_KEY else 'not found')

NEWS API KEY LOADED


##Install Libraries
We install all required Python libraries needed for this project.
yfinance fetches real stock prices, requests calls NewsAPI for headlines,
pandas handles data cleaning, and transformers loads the FinBERT AI model.





In [ ]:
import os
os.makedirs('/content/stock_project/data', exist_ok=True)
os.makedirs('/content/stock_project/outputs', exist_ok=True)
print("foler created")

foler created


In [ ]:
import yfinance as yf
import pandas as pd

##Load API Keys
We securely load our API keys from Colab Secrets — this keeps keys
out of the code so they are safe to share on GitHub or portfolio.

In [ ]:
stocks = ['AAPL', 'GOOGL' , 'MSFT' , 'AMZN' , 'TSLA']
all_prices = []

for ticker in stocks:
  raw = yf.download(ticker, period='30d', interval='1d', auto_adjust=True, progress=False)

  close = raw['Close'].squeeze()
  volume = raw['Volume'].squeeze()

  temp = pd.DataFrame({
      'Ticker':ticker,
      'Date':raw.index.date,
      'close':close.values.round(2),
      'volume':volume.values.astype(int)
  })

  all_prices.append(temp)

prices_df = pd.concat(all_prices, axis=0, ignore_index=True)

print(prices_df.head(10))
print(f'\nTotal rows: {len(prices_df)}')
print(f'Stocks covered: {prices_df['Ticker'].unique()}')

  Ticker        Date   close    volume
0   AAPL  2026-03-27  248.80  47900000
1   AAPL  2026-03-30  246.63  39446200
2   AAPL  2026-03-31  253.79  49598100
3   AAPL  2026-04-01  255.63  40059400
4   AAPL  2026-04-02  255.92  31289400
5   AAPL  2026-04-06  258.86  29329900
6   AAPL  2026-04-07  253.50  62148000
7   AAPL  2026-04-08  258.90  41032800
8   AAPL  2026-04-09  260.49  28121600
9   AAPL  2026-04-10  260.48  31291500

Total rows: 150
Stocks covered: ['AAPL' 'GOOGL' 'MSFT' 'AMZN' 'TSLA']


##Fetch Real Stock Price Data
Using yfinance we pull 30 days of real closing prices and volume
for 5 major stocks — AAPL, GOOGL, MSFT, AMZN, TSLA — directly
from Yahoo Finance. No manual download needed.

In [ ]:
import requests

def get_headlines(ticker_symbol, ticker_name, api_key, num_articles=10):
    url = 'https://newsapi.org/v2/everything'
    params = {
        'q': ticker_name,
        'language': 'en',
        'sortBy': 'publishedAt',
        'pageSize': num_articles,
        'apiKey': api_key
    }
    response = requests.get(url, params=params)
    data = response.json()

    headlines = []
    for article in data.get('articles', []):
        headlines.append({
            'Ticker': ticker_symbol,
            'Date': article['publishedAt'][:10],
            'Headline': article['title']
        })
    return headlines

ticker_names = {
    'AAPL': 'Apple stock',
    'GOOGL': 'Google stock',
    'MSFT': 'Microsoft stock',
    'AMZN': 'Amazon stock',
    'TSLA': 'Tesla stock'
}

all_headlines = []

for ticker, name in ticker_names.items():
    headlines = get_headlines(ticker, name, NEWS_API_KEY, num_articles=10)
    all_headlines.extend(headlines)
    print(f"✓ {ticker}: {len(headlines)} headlines fetched")

headlines_df = pd.DataFrame(all_headlines)
print("\n--- Sample Headlines ---")
print(headlines_df.head(10))
print(f"\nTotal headlines: {len(headlines_df)}")

✓ AAPL: 10 headlines fetched
✓ GOOGL: 10 headlines fetched
✓ MSFT: 10 headlines fetched
✓ AMZN: 9 headlines fetched
✓ TSLA: 10 headlines fetched

--- Sample Headlines ---
  Ticker        Date                                           Headline
0   AAPL  2026-05-09  Paul Tudor Jones Warns Trump-Era Market Boom C...
1   AAPL  2026-05-09  Analysts turn heads with AMD stock forecast af...
2   AAPL  2026-05-09  25 Health Brand Campaigns That Redefined Digit...
3   AAPL  2026-05-09  Apple Looking to Add MacBook Neo To Deal with ...
4   AAPL  2026-05-09             Samsung Achieves $1 Trillion Valuation
5   AAPL  2026-05-09  Intel soars on Apple chip deal, lifting S&P 50...
6   AAPL  2026-05-09                  PipeDream on the Acorn Archimedes
7   AAPL  2026-05-09  Intel CEO who won over Trump and Musk now need...
8   AAPL  2026-05-09  (Video) Arne Slot will have loved what Rio Ngu...
9   AAPL  2026-05-09  PayPal Faces a Brutal Reality: 3 Real Problems...

Total headlines: 49


##Fetch Live News Headlines
Using NewsAPI we fetch the 10 most recent financial news headlines
for each stock. This gives us real-world sentiment signals to
analyze against price movements.

In [ ]:
from transformers import pipeline

print("Loading FinBERT model... (takes 1-2 mins first time)")
finbert = pipeline("text-classification", model="ProsusAI/finbert")
print("✓ FinBERT loaded!")

# Run sentiment on every headline
print("\nAnalyzing sentiment...")

sentiments = []
for _, row in headlines_df.iterrows():
    result = finbert(row['Headline'][:512])[0]  # 512 char limit
    sentiments.append({
        'Ticker': row['Ticker'],
        'Date': row['Date'],
        'Headline': row['Headline'],
        'Sentiment': result['label'],
        'Confidence': round(result['score'], 2)
    })

sentiment_df = pd.DataFrame(sentiments)
print("✓ Sentiment analysis done!")
print("\n--- Sample Results ---")
print(sentiment_df.head(10))
print(f"\nSentiment breakdown:\n{sentiment_df['Sentiment'].value_counts()}")

Loading FinBERT model... (takes 1-2 mins first time)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/758 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

...

BertForSequenceClassification LOAD REPORT from: ProsusAI/finbert
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/252 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

✓ FinBERT loaded!

Analyzing sentiment...
✓ Sentiment analysis done!

--- Sample Results ---
  Ticker        Date                                           Headline  \
0   AAPL  2026-05-09  Paul Tudor Jones Warns Trump-Era Market Boom C...   
1   AAPL  2026-05-09  Analysts turn heads with AMD stock forecast af...   
2   AAPL  2026-05-09  25 Health Brand Campaigns That Redefined Digit...   
3   AAPL  2026-05-09  Apple Looking to Add MacBook Neo To Deal with ...   
4   AAPL  2026-05-09             Samsung Achieves $1 Trillion Valuation   
5   AAPL  2026-05-09  Intel soars on Apple chip deal, lifting S&P 50...   
6   AAPL  2026-05-09                  PipeDream on the Acorn Archimedes   
7   AAPL  2026-05-09  Intel CEO who won over Trump and Musk now need...   
8   AAPL  2026-05-09  (Video) Arne Slot will have loved what Rio Ngu...   
9   AAPL  2026-05-09  PayPal Faces a Brutal Reality: 3 Real Problems...   

  Sentiment  Confidence  
0   neutral        0.70  
1  negative        0.61  
2  

##AI Sentiment Analysis with FinBERT
We apply FinBERT — a transformer-based NLP model trained specifically
on financial text — to classify each headline as positive, negative,
or neutral with a confidence score. This is the core AI integration
of the project.

In [ ]:
import sqlite3

conn = sqlite3.connect('stock_sentiment.db')

prices_df.to_sql('stock_prices', conn, if_exists='replace', index=False)
sentiment_df.to_sql('sentiment', conn, if_exists='replace', index=False)

# Get latest price per ticker and join on ticker only
query = """
SELECT
    s.Ticker,
    s.Date,
    s.Headline,
    s.Sentiment,
    s.Confidence,
    p.Close,
    p.Volume
FROM sentiment s
LEFT JOIN (
    SELECT Ticker, Close, Volume
    FROM stock_prices
    WHERE (Ticker, Date) IN (
        SELECT Ticker, MAX(Date)
        FROM stock_prices
        GROUP BY Ticker
    )
) p ON s.Ticker = p.Ticker
"""

final_df = pd.read_sql_query(query, conn)
conn.close()

print(f"Total rows: {len(final_df)}")
print("\n--- Final Joined Table ---")
print(final_df.head(10))
print("\nNull check:")
print(final_df.isnull().sum())

Total rows: 49

--- Final Joined Table ---
  Ticker        Date                                           Headline  \
0   AAPL  2026-05-09  Paul Tudor Jones Warns Trump-Era Market Boom C...   
1   AAPL  2026-05-09  Analysts turn heads with AMD stock forecast af...   
2   AAPL  2026-05-09  25 Health Brand Campaigns That Redefined Digit...   
3   AAPL  2026-05-09  Apple Looking to Add MacBook Neo To Deal with ...   
4   AAPL  2026-05-09             Samsung Achieves $1 Trillion Valuation   
5   AAPL  2026-05-09  Intel soars on Apple chip deal, lifting S&P 50...   
6   AAPL  2026-05-09                  PipeDream on the Acorn Archimedes   
7   AAPL  2026-05-09  Intel CEO who won over Trump and Musk now need...   
8   AAPL  2026-05-09  (Video) Arne Slot will have loved what Rio Ngu...   
9   AAPL  2026-05-09  PayPal Faces a Brutal Reality: 3 Real Problems...   

  Sentiment  Confidence   Close    Volume  
0   neutral        0.70  293.32  52631200  
1  negative        0.61  293.32  52631200  

##SQL Database and Data Join
We store both datasets in a local SQLite database and join sentiment
scores with stock prices using SQL — simulating a real analyst workflow
where multiple data sources are merged for analysis.

In [ ]:
# Add a sentiment score column (numeric version for Power BI charts)
final_df['Sentiment_Score'] = final_df['Sentiment'].map({
    'positive': 1,
    'neutral': 0,
    'negative': -1
})

# Summary table per stock (useful for Power BI)
summary_df = final_df.groupby('Ticker').agg(
    Latest_Price=('Close', 'first'),
    Total_Headlines=('Headline', 'count'),
    Positive=('Sentiment', lambda x: (x == 'positive').sum()),
    Neutral=('Sentiment', lambda x: (x == 'neutral').sum()),
    Negative=('Sentiment', lambda x: (x == 'negative').sum()),
    Avg_Confidence=('Confidence', 'mean')
).reset_index()

summary_df['Avg_Confidence'] = summary_df['Avg_Confidence'].round(2)
summary_df['Sentiment_Score'] = (summary_df['Positive'] - summary_df['Negative']) / summary_df['Total_Headlines']
summary_df['Sentiment_Score'] = summary_df['Sentiment_Score'].round(2)

print("--- Stock Summary ---")
print(summary_df)

# Export both tables to CSV
final_df.to_csv('stock_sentiment_final.csv', index=False)
summary_df.to_csv('stock_summary.csv', index=False)

print("\n✓ Files saved! Now download them:")

# Download to your Windows machine
from google.colab import files
files.download('stock_sentiment_final.csv')
files.download('stock_summary.csv')

--- Stock Summary ---
  Ticker  Latest_Price  Total_Headlines  Positive  Neutral  Negative  \
0   AAPL        293.32               10         3        5         2   
1   AMZN        272.68                9         3        5         1   
2  GOOGL        400.80               10         3        6         1   
3   MSFT        415.12               10         3        5         2   
4   TSLA        428.35               10         3        6         1   

   Avg_Confidence  Sentiment_Score  
0            0.80             0.10  
1            0.86             0.22  
2            0.80             0.20  
3            0.84             0.10  
4            0.85             0.20  

✓ Files saved! Now download them:


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

##Export for Power BI Dashboard
We create a summary table aggregating sentiment per stock and export
both the detailed and summary data as CSV files — ready to load into
Power BI for the final interactive dashboard.